# Lightweight options research
This notebook runs the cached engine and displays native/USD equity, option-only P&L, drawdowns, fees, and margin.

In [ ]:
from pathlib import Path
import pandas as pd
import plotly.express as px
from lightweight_backtester.cli import run_config
from lightweight_backtester.config import ResearchConfig
ROOT = Path('..').resolve()
cfg = ResearchConfig.from_toml(ROOT / 'configs/btc_weekly_straddle_smoke.toml')
result = run_config(cfg, ROOT / 'results/notebook_smoke')
result.metrics

In [ ]:
eq = pd.DataFrame([x.__dict__ for x in result.equity])
eq['date'] = pd.to_datetime(eq.ts_ns, unit='ns', utc=True)
eq['native_drawdown'] = eq.native_equity / eq.native_equity.cummax() - 1
eq['usd_drawdown'] = eq.usd_equity / eq.usd_equity.cummax() - 1
px.line(eq, x='date', y=['native_equity','option_pnl_native'], title='Native and option-only equity').show()
px.line(eq, x='date', y='usd_equity', title='USD-marked equity').show()
px.line(eq, x='date', y=['native_drawdown','usd_drawdown'], title='Drawdowns').show()
px.line(eq, x='date', y=['initial_margin_native','maintenance_margin_native'], title='Reported margin').show()

In [ ]:
trades = pd.DataFrame(result.trade_ledger)
legs = pd.DataFrame(result.leg_ledger)
display(trades.head(), legs.head())
px.bar(trades, x=pd.to_datetime(trades.exit_ts_ns, unit='ns', utc=True), y='net_pnl_native', title='Trade P&L').show()